# Laboratory 6 — Deploying the Titanic Survival Predictor as a FastAPI Service

**Course:** Introduction to Artificial Intelligence (AI–101L) · **Instructor:** Dr. Muhammad Awais
**Duration:** 3 hours · **Block:** Phase B — Classical ML and Serving · **Mapped CLO:** CLO–4

---

## Learning Objectives

On completion of this laboratory you will be able to:

1. Persist scikit-learn preprocessing artefacts (label encoders or mappings) **alongside** the trained model.
2. Build a FastAPI application that loads its artefacts **once at start-up** and serves predictions over a JSON API.
3. Create **Pydantic** request and response schemas for input validation and automatic API documentation.
4. Add the operational endpoints `/health` and `/metadata`, and handle errors with meaningful HTTP status codes.
5. Run and test the API locally, and containerise it with Docker.

## Background

This laboratory converts the **application phase** of the trained Titanic model (Laboratory 5)
from an interactive script — one that reads `input()` values, encodes them and calls
`model.predict` — into a **networked API** that can be queried by a front end or by another
service.

Such an API must satisfy three requirements that a notebook never has to:

1. It must **validate** its inputs and return meaningful HTTP status codes when they are wrong.
2. It must use the **exact same encodings** the model saw during training.
3. It must **load artefacts once at start-up**, not on every request, or throughput collapses.

```
Client  ->  Pydantic     ->  Label encoders  ->  SVC model     ->  PredictResponse
browser     validation       training-time       .predict()        label + SURVIVED
curl        422 on a         mappings                              JSON response
front end   bad body         400 on unknown                        back to client
                             category
```

> **Instructor note.** This notebook drives the *real* `main.py` that already sits in this folder.
> Section 4 starts a live Uvicorn server **from inside the notebook** on a background thread, so
> the whole lab can be demonstrated without a second terminal. Students should still run the
> `uvicorn` command themselves at least once.

---
# Task 6.1 — Setup and artefact inventory

Install the serving stack, then confirm that every artefact Laboratory 5 produced is present in
this folder.

In [1]:
!pip install fastapi "uvicorn[standard]" pandas numpy scikit-learn joblib requests


   ---------------------------------------- 0/3 [opentelemetry-api]
   -------------------------- ------------- 2/3 [fastapi]
   -------------------------- ------------- 2/3 [fastapi]
   ---------------------------------------- 3/3 [fastapi]



In [3]:
import os
import pickle
import numpy as np
import pandas as pd

# The five artefacts the service needs: one model + four label encoders.
ARTEFACTS = {
    "model":       "svc_trained_model.pkl",
    "PClass":      "pclass_encoder.pkl",
    "Gender":      "gender_encoder.pkl",
    "Sibling":     "sibling_encoder.pkl",
    "Embarked":    "embarked_encoder.pkl",
}

print(f"{'ARTEFACT':<12} {'FILE':<28} {'STATUS':<10} SIZE")
print("-" * 62)
for name, path in ARTEFACTS.items():
    ok = os.path.exists(path)
    size = f"{os.path.getsize(path):,} B" if ok else "-"
    print(f"{name:<12} {path:<28} {'FOUND' if ok else 'MISSING':<10} {size}")

ARTEFACT     FILE                         STATUS     SIZE
--------------------------------------------------------------
model        svc_trained_model.pkl        FOUND      4,850 B
PClass       pclass_encoder.pkl           FOUND      268 B
Gender       gender_encoder.pkl           FOUND      259 B
Sibling      sibling_encoder.pkl          FOUND      270 B
Embarked     embarked_encoder.pkl         FOUND      282 B


## Task 6.1a — Persist the preprocessing artefacts

If any encoder is missing, go back to your Laboratory 5 training script and add the block below
**after the encoders are fitted**. This guarantees the API applies identical encodings at
inference time.

> ### Alternative, and better
>
> Save a single `sklearn.Pipeline` containing the preprocessing **and** the model, serialised with
> `joblib`. One artefact cannot fall out of step with itself, whereas five separate pickles can —
> and eventually will. Exercise 1 asks you to make this change and argue the trade-off.

In [4]:
# ============================================================================
# Add this to the END of your Lab 05 training script (shown here for reference).
# ============================================================================

# import pickle
#
# with open('pclass_encoder.pkl',  'wb') as f: pickle.dump(pclass_label_encoder,  f)
# with open('gender_encoder.pkl',  'wb') as f: pickle.dump(gender_label_encoder,  f)
# with open('sibling_encoder.pkl', 'wb') as f: pickle.dump(sibling_label_encoder, f)
# with open('embarked_encoder.pkl','wb') as f: pickle.dump(embarked_label_encoder,f)
#
# # Save the trained model too
# with open('svc_trained_model.pkl','wb') as f: pickle.dump(svc_model, f)

print("Artefacts for this lab already exist in the folder -- see the table above.")

Artefacts for this lab already exist in the folder -- see the table above.


In [5]:
# Load the encoders and read back the EXACT category vocabulary the model was
# trained on. This vocabulary is the API contract: anything outside it is a 400.

encoders = {}
for field in ["PClass", "Gender", "Sibling", "Embarked"]:
    with open(ARTEFACTS[field], "rb") as f:
        encoders[field] = pickle.load(f)

print("Allowed categories, straight from the fitted encoders:\n")
for field, enc in encoders.items():
    classes = list(enc.classes_)
    codes = list(enc.transform(classes))
    print(f"{field:<10}: {classes}")
    print(f"{'':<10}  encoded as {codes}\n")

Allowed categories, straight from the fitted encoders:

PClass    : ['First', 'Second', 'Third']
            encoded as [np.int64(0), np.int64(1), np.int64(2)]

Gender    : ['Female', 'Male']
            encoded as [np.int64(0), np.int64(1)]

Sibling   : ['One', 'Three', 'Two', 'Zero']
            encoded as [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]

Embarked  : ['Cherbourg', 'Queenstown', 'Southampton']
            encoded as [np.int64(0), np.int64(1), np.int64(2)]



C:\Users\dell pc\anaconda3\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [6]:
# Load the model and confirm what it expects.
with open(ARTEFACTS["model"], "rb") as f:
    model = pickle.load(f)

print("Model type          :", type(model).__name__)
print("Expects n features  :", getattr(model, "n_features_in_", "unknown"))
if hasattr(model, "feature_names_in_"):
    print("Feature names/order :", list(model.feature_names_in_))
print("Has predict_proba   :", hasattr(model, "predict_proba"))

Model type          : SVC
Expects n features  : 4
Feature names/order : ['PClass', 'Gender', 'Sibling', 'Embarked']
Has predict_proba   : False


C:\Users\dell pc\anaconda3\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.6.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


---
# Task 6.2 — Create the Pydantic schemas

Define the request and response types. `Literal` restricts each field to the exact set of
categories the encoders know, which both **tightens validation** and **documents the contract**
in the generated Swagger page.

The cell below *writes* `app/schemas.py` to disk — it is a real file, not just a demonstration.

In [ ]:
os.makedirs("app", exist_ok=True)

schemas_py = '''# app/schemas.py
from pydantic import BaseModel
from typing import Literal


class PredictRequest(BaseModel):
    PClass:   Literal["First", "Second", "Third"]
    Gender:   Literal["Male", "Female"]
    Sibling:  Literal["Zero", "One", "Two", "Three"]
    Embarked: Literal["Southampton", "Cherbourg", "Queenstown"]


class PredictResponse(BaseModel):
    predicted_label: int
    prediction: Literal["SURVIVED", "NOT SURVIVED"]
'''

with open("app/schemas.py", "w", encoding="utf-8") as f:
    f.write(schemas_py)

print("Wrote app/schemas.py\n")
print(schemas_py)

In [7]:
# Try the schema out in isolation, before any server is involved.
from pydantic import BaseModel, ValidationError
from typing import Literal


class PredictRequest(BaseModel):
    PClass:   Literal["First", "Second", "Third"]
    Gender:   Literal["Male", "Female"]
    Sibling:  Literal["Zero", "One", "Two", "Three"]
    Embarked: Literal["Southampton", "Cherbourg", "Queenstown"]


# 1. A valid body parses cleanly
good = PredictRequest(PClass="First", Gender="Male",
                      Sibling="Zero", Embarked="Southampton")
print("VALID  ->", good)

# 2. An invalid category is rejected BEFORE it ever reaches your model
try:
    PredictRequest(PClass="Fourth", Gender="Male",
                   Sibling="Zero", Embarked="Southampton")
except ValidationError as e:
    print("\nINVALID -> rejected by Pydantic:")
    print(e.errors()[0]["msg"])

VALID  -> PClass='First' Gender='Male' Sibling='Zero' Embarked='Southampton'

INVALID -> rejected by Pydantic:
Input should be 'First', 'Second' or 'Third'


---
# Task 6.3 — The FastAPI application

The application must:

- load artefacts **once** at start-up,
- expose `/predict`, `/metadata` and `/health`,
- convert an unknown-category error from a `LabelEncoder` into an **HTTP 400** with a helpful message.

| Endpoint | Method | Purpose |
|---|---|---|
| `/predict` | POST | Validates, encodes, predicts, returns the label and its human-readable form |
| `/metadata` | GET | Returns the allowed categorical values, so a front end can populate its dropdowns from the server rather than hard-coding them |
| `/health` | GET | A cheap readiness check for load balancers, containers and uptime monitors |
| `/docs` | GET | Swagger UI, generated automatically from the Pydantic schemas |

The reference implementation for this laboratory is **`main.py`**, already present in this
folder. Read it before running it.

In [8]:
# Read the service you are about to run. Never start a server you have not read.
with open("main.py", encoding="utf-8") as f:
    source = f.read()

print(f"main.py is {len(source.splitlines())} lines.\n")
print("Endpoints defined:")
for line in source.splitlines():
    s = line.strip()
    if s.startswith("@app."):
        print("   ", s)

FileNotFoundError: [Errno 2] No such file or directory: 'main.py'

### The canonical minimal service

For reference, this is the smallest complete version of the same service — the one printed in
the manual. Compare it against `main.py` and note what the fuller version adds
(category validation, confidence, echo of the encoded input).

In [ ]:
reference_main = '''# app/main.py  -- the minimal reference implementation
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
import pickle
import pandas as pd
import numpy as np
import logging
from .schemas import PredictRequest, PredictResponse

app = FastAPI(title="Titanic Survival Predictor")

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# CORS -- required by the Next.js front end of Lab 12. Tighten in production.
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

MODEL_PATH        = "svc_trained_model.pkl"
PCLASS_ENC_PATH   = "pclass_encoder.pkl"
GENDER_ENC_PATH   = "gender_encoder.pkl"
SIBLING_ENC_PATH  = "sibling_encoder.pkl"
EMBARKED_ENC_PATH = "embarked_encoder.pkl"


def load_artifacts():
    global model, pclass_le, gender_le, sibling_le, embarked_le
    with open(MODEL_PATH, "rb") as f:        model      = pickle.load(f)
    with open(PCLASS_ENC_PATH, "rb") as f:   pclass_le  = pickle.load(f)
    with open(GENDER_ENC_PATH, "rb") as f:   gender_le  = pickle.load(f)
    with open(SIBLING_ENC_PATH, "rb") as f:  sibling_le = pickle.load(f)
    with open(EMBARKED_ENC_PATH, "rb") as f: embarked_le = pickle.load(f)


@app.on_event("startup")
def startup_event():
    load_artifacts()                 # ONCE, not per request
    logger.info("Model and encoders loaded")


@app.get("/health")
def health():
    return {"status": "ok"}


@app.get("/metadata")
def metadata():
    return {
        "PClass":   list(pclass_le.classes_),
        "Gender":   list(gender_le.classes_),
        "Sibling":  list(sibling_le.classes_),
        "Embarked": list(embarked_le.classes_),
    }


@app.post("/predict", response_model=PredictResponse)
def predict(req: PredictRequest):
    try:
        df = pd.DataFrame([{
            "PClass": req.PClass, "Gender": req.Gender,
            "Sibling": req.Sibling, "Embarked": req.Embarked,
        }])

        # Encode with the SAME encoders the model was trained on
        df["PClass"]   = pclass_le.transform(df["PClass"])
        df["Gender"]   = gender_le.transform(df["Gender"])
        df["Sibling"]  = sibling_le.transform(df["Sibling"])
        df["Embarked"] = embarked_le.transform(df["Embarked"])

        features   = df[["PClass", "Gender", "Sibling", "Embarked"]].to_numpy()
        pred_value = int(np.ravel(model.predict(features))[0])
        label      = "SURVIVED" if pred_value == 1 else "NOT SURVIVED"

        return PredictResponse(predicted_label=pred_value, prediction=label)

    except ValueError as ve:
        # Unknown category passed to LabelEncoder.transform
        raise HTTPException(status_code=400, detail=str(ve))
    except Exception as e:
        logger.exception("Prediction error")
        raise HTTPException(status_code=500, detail=str(e))
'''

with open("app/main_reference.py", "w", encoding="utf-8") as f:
    f.write(reference_main)

print("Wrote app/main_reference.py for study.")
print("The service actually started below is the fuller main.py in this folder.")

---
# Task 6.4 — Run the API

In a normal workflow you start the server from a terminal:

```bash
uvicorn main:app --reload --host 0.0.0.0 --port 8000
```

Swagger UI is then at **<http://localhost:8000/docs>**.

For the guided walkthrough below we start the *same* application on a **background thread**, so
that the notebook can call it directly. Run the terminal command yourself at least once as
well — that is what you will do in Laboratory 15.

In [ ]:
import threading
import time
import os
import sys

import uvicorn
import requests

# Jupyter puts the notebook's folder on sys.path automatically; this line makes
# the notebook work identically when exported and run as a plain script.
sys.path.insert(0, os.getcwd())

from main import app as titanic_app   # import the real service in this folder

HOST, PORT = "127.0.0.1", 8000
BASE_URL = f"http://{HOST}:{PORT}"


def _serve():
    uvicorn.run(titanic_app, host=HOST, port=PORT, log_level="warning")


server_thread = threading.Thread(target=_serve, daemon=True)
server_thread.start()

# Wait for the server to come up, using /health -- which is exactly what
# a container orchestrator does in Lab 15.
for attempt in range(30):
    try:
        r = requests.get(f"{BASE_URL}/health", timeout=1)
        if r.status_code == 200:
            print(f"Server is up after {attempt + 1} attempt(s): {r.json()}")
            break
    except requests.exceptions.RequestException:
        time.sleep(0.5)
else:
    print("Server did not start. Run uvicorn in a terminal instead.")

In [ ]:
import json   # used to pretty-print JSON responses throughout this section

## Task 6.4a — `/metadata`: let the server own the vocabulary

The reference `main.py` in this folder exposes `/health` and `/predict` but **not** `/metadata`.
That is a real gap, and a good illustration of why the endpoint matters:

> Without `/metadata`, the front end of Laboratory 12 has to **hard-code** the four dropdown lists.
> The day someone retrains the model with a new category, the UI silently offers the wrong choices
> and every such request fails with a 400. Serving the vocabulary from the same process that owns
> the encoders makes that impossible.

The cell below **mounts the missing route onto the already-running application**. Adding a route
after start-up like this is a demonstration technique — for your submission, move this function
into `main.py` where it belongs.

In [ ]:
# Does the running service already expose /metadata?
existing = {getattr(r, "path", None) for r in titanic_app.routes}
print("Routes before:", sorted(p for p in existing if p and not p.startswith("/openapi")))

if "/metadata" not in existing:

    @titanic_app.get("/metadata")
    def metadata():
        """Return the exact category vocabulary the encoders were fitted on."""
        return {field: list(enc.classes_) for field, enc in encoders.items()}

    print("\nMounted /metadata onto the running app.")
else:
    print("\n/metadata already present.")

# Verify it over HTTP, the way a front end would call it
r = requests.get(f"{BASE_URL}/metadata", timeout=10)
print("Status:", r.status_code)
print(json.dumps(r.json(), indent=2))

In [ ]:
# GET /health -- the readiness probe
r = requests.get(f"{BASE_URL}/health")
print("Status:", r.status_code)
print("Body  :", r.json())

In [ ]:
# GET /docs -- confirm the interactive documentation was generated
r = requests.get(f"{BASE_URL}/docs")
print("Swagger UI status:", r.status_code)
print("Open this URL in your browser:", f"{BASE_URL}/docs")

# The machine-readable contract behind that page:
schema = requests.get(f"{BASE_URL}/openapi.json").json()
print("\nPaths exposed by the service:")
for path, methods in schema["paths"].items():
    print("   ", ", ".join(m.upper() for m in methods), path)

## Task 6.5 — Call `/predict`

The equivalent `curl` command is:

```bash
curl -X POST "http://localhost:8000/predict" \
  -H "Content-Type: application/json" \
  -d '{"PClass":"First","Gender":"Male","Sibling":"Zero","Embarked":"Southampton"}'
```

In [ ]:
payload = {
    "PClass":   "First",
    "Gender":   "Male",
    "Sibling":  "Zero",
    "Embarked": "Southampton",
}

r = requests.post(f"{BASE_URL}/predict", json=payload, timeout=10)

print("Status:", r.status_code)
print("Response:")
import json
print(json.dumps(r.json(), indent=2))

In [ ]:
# Run a small batch of scenarios and tabulate them -- this is the evidence
# your one-page report needs.
#
# NOTE: this SVC was trained without probability=True, so it has no
# predict_proba and main.py falls back to a fixed confidence of 0.85.
# A constant 'confidence' is worse than none at all -- see Exercise 5.

scenarios = [
    {"PClass": "First",  "Gender": "Female", "Sibling": "Zero", "Embarked": "Cherbourg"},
    {"PClass": "First",  "Gender": "Male",   "Sibling": "Zero", "Embarked": "Southampton"},
    {"PClass": "Third",  "Gender": "Male",   "Sibling": "Zero", "Embarked": "Southampton"},
    {"PClass": "Third",  "Gender": "Female", "Sibling": "Two",  "Embarked": "Queenstown"},
    {"PClass": "Second", "Gender": "Female", "Sibling": "One",  "Embarked": "Southampton"},
]

rows = []
for s in scenarios:
    resp = requests.post(f"{BASE_URL}/predict", json=s, timeout=10)
    body = resp.json()
    rows.append({
        **s,
        "status": resp.status_code,
        "prediction": body.get("prediction_label", body.get("prediction")),
        "confidence": round(body.get("confidence", float("nan")), 3),
    })

results = pd.DataFrame(rows)
display(results)

---
# Task 6.6 — Error handling: two different failures, two different codes

> ### Two different failure codes, two different causes
>
> A body that is **missing a field or has the wrong type** never reaches your handler: Pydantic
> rejects it with **422**.
>
> A **well-formed body carrying a category the encoder has never seen** reaches the handler and
> raises `ValueError`, which you translate into **400**.
>
> Being able to explain that difference is part of the assessment.

In [ ]:
# CASE 1 -- unknown category in a well-formed body  -> expect 400
bad_category = {
    "PClass":   "Fourth",          # there was no fourth class on the Titanic
    "Gender":   "Male",
    "Sibling":  "Zero",
    "Embarked": "Southampton",
}

r = requests.post(f"{BASE_URL}/predict", json=bad_category, timeout=10)
print("Status:", r.status_code, "(expected 400)")
print(json.dumps(r.json(), indent=2)[:600])

In [ ]:
# CASE 2 -- malformed body, a required field is missing  -> expect 422
missing_field = {
    "PClass":  "First",
    "Gender":  "Male",
    # "Sibling" omitted on purpose
    "Embarked": "Southampton",
}

r = requests.post(f"{BASE_URL}/predict", json=missing_field, timeout=10)
print("Status:", r.status_code, "(expected 422)")
print(json.dumps(r.json(), indent=2)[:600])

In [ ]:
# CASE 3 -- wrong type entirely  -> expect 422
wrong_type = {"PClass": 1, "Gender": "Male", "Sibling": "Zero", "Embarked": "Southampton"}

r = requests.post(f"{BASE_URL}/predict", json=wrong_type, timeout=10)
print("Status:", r.status_code)
print(json.dumps(r.json(), indent=2)[:400])

In [ ]:
# Summarise the contract you have just demonstrated.
summary = pd.DataFrame([
    {"case": "valid request",              "expected": 200, "cause": "-"},
    {"case": "unknown category",           "expected": 400, "cause": "LabelEncoder.transform raises ValueError in the handler"},
    {"case": "missing required field",     "expected": 422, "cause": "Pydantic rejects the body before the handler runs"},
    {"case": "wrong field type",           "expected": 422, "cause": "Pydantic rejects the body before the handler runs"},
    {"case": "artefacts not loaded",       "expected": 500, "cause": "Server-side fault, not the client's mistake"},
])
display(summary)

---
# Task 6.7 — Measure start-up cost vs per-request cost

This is *why* artefacts are loaded once at start-up. Measure both and put the numbers in your
report.

In [ ]:
import time

# Cost of loading the artefacts from disk (paid ONCE at start-up)
t0 = time.perf_counter()
with open(ARTEFACTS["model"], "rb") as f:
    _ = pickle.load(f)
for field in ["PClass", "Gender", "Sibling", "Embarked"]:
    with open(ARTEFACTS[field], "rb") as f:
        _ = pickle.load(f)
load_ms = (time.perf_counter() - t0) * 1000

# Cost of one prediction over HTTP (paid on EVERY request)
latencies = []
for _ in range(20):
    t0 = time.perf_counter()
    requests.post(f"{BASE_URL}/predict", json=payload, timeout=10)
    latencies.append((time.perf_counter() - t0) * 1000)

print(f"Artefact load (once at start-up): {load_ms:8.2f} ms")
print(f"Request latency, mean of 20     : {np.mean(latencies):8.2f} ms")
print(f"Request latency, p95            : {np.percentile(latencies, 95):8.2f} ms")
print()
print(f"Loading per request would add roughly {load_ms / np.mean(latencies):.1f}x to every call.")

---
# Task 6.8 — Containerise the service

Package the application and its artefacts into a Docker image so that it runs identically on any
machine. The cell below writes `requirements.txt` and `Dockerfile`.

In [ ]:
requirements_txt = """fastapi
uvicorn[standard]
pandas
numpy
scikit-learn
joblib
"""

dockerfile = """FROM python:3.11-slim
WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY main.py .
COPY *.pkl ./

EXPOSE 8000
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
"""

with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements_txt)
with open("Dockerfile", "w", encoding="utf-8") as f:
    f.write(dockerfile)

print("Wrote requirements.txt and Dockerfile\n")
print(dockerfile)

In [ ]:
# ============================================================================
# RUN IN A TERMINAL (Docker Desktop must be running).
# ============================================================================

# docker build -t titanic-fastapi:latest .
# docker run -p 8000:8000 titanic-fastapi:latest
#
# Then, from another terminal:
# curl http://localhost:8000/health

print("Build and run the container, then paste the /health response in your report.")

---
# In-Lab Exercises

**Exercise 1 — Bundle preprocessing.** Re-train and export a `sklearn.Pipeline` that contains the
encoders **and** the classifier, then modify the API to load only that single artefact. Explain
the advantages and disadvantages of each approach.

**Exercise 2 — Unknown category handling.** Change the API to accept unknown categories by mapping
them to a reserved index (for example $-1$), document the behaviour, and measure how the
predictions differ.

**Exercise 3 — Versioning.** Add a `/version` endpoint returning `model_version` and artefact
checksums, and describe how you would use it in A/B testing.

**Exercise 4 — Monitoring.** Add basic Prometheus metrics — request count, latency and error
count — to the FastAPI app.

**Exercise 5 — Honest confidence.** `svc_trained_model.pkl` was trained without `probability=True`, so it cannot report a real confidence and the service returns a hard-coded `0.85`. Retrain the SVC with `probability=True`, return the true `predict_proba` maximum, and explain why a *constant* confidence value is more dangerous than showing no confidence at all.

In [ ]:
# --- Exercise 3 scaffold: artefact checksums for a /version endpoint -----
import hashlib


def file_sha256(path, chunk=65536):
    """Return the SHA-256 of a file, read in chunks so large models are safe."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


version_info = {
    "model_version": "svc-1.0.0",
    "artifact_checksums": {name: file_sha256(path)[:16]
                           for name, path in ARTEFACTS.items()
                           if os.path.exists(path)},
}

print(json.dumps(version_info, indent=2))
print()
print("Add this dict as the body of a GET /version endpoint in main.py.")
print("In A/B testing, the checksum proves WHICH artefacts served a given request.")

---
# Home Assignment

Write an automated test suite with `pytest` and `fastapi.testclient.TestClient` covering:

- a successful prediction,
- a **422** from a missing field,
- a **400** from an unknown category,
- the shape of the `/metadata` response,
- `/health`.

The suite must run **without a server process**. Commit it as `tests/test_api.py`; Laboratory 13
will run it in continuous integration.

The cell below writes a working starting point and runs it.

In [ ]:
!pip install pytest httpx

# If an unrelated global pytest plugin (langsmith, for example) crashes the
# collector on your machine, disable it rather than uninstalling it:
#     !pytest tests/test_api.py -v -p no:langsmith_plugin
!pytest tests/test_api.py -v

In [ ]:
!pip install pytest httpx
!pytest tests/test_api.py -v

---
# Deliverables

| File | Contents |
|---|---|
| `main.py` (or `app/main.py`) | The running service |
| `app/schemas.py` | Pydantic request/response models |
| `*.pkl` | `svc_trained_model.pkl` and the four encoder pickles |
| `requirements.txt` | Pinned serving dependencies |
| `Dockerfile` | The container definition |
| `tests/test_api.py` | The pytest suite |
| `lab06/report.md` | Artefact saving strategy, the `/predict` contract, and results from **at least three** example requests |

## Assessment Rubric

| Criterion | Weight | Excellent performance |
|---|---|---|
| Correctness of implementation | 35 % | All endpoints work; artefacts load once at start-up |
| Methodological soundness | 25 % | Inference uses the *same* encoders as training; 400 vs 422 handled correctly |
| Analysis and interpretation | 20 % | Latency and start-up cost measured and explained |
| Code quality and reproducibility | 10 % | Containerised; tests run without a server |
| Report and demonstration | 10 % | Live demonstration of all endpoints including failures |

## References

- FastAPI documentation — <https://fastapi.tiangolo.com/>
- scikit-learn persistence — <https://scikit-learn.org/stable/modules/model_persistence.html>
- Uvicorn — <https://www.uvicorn.org/>

---

**Next laboratory:** Lab 07 — Deep Learning for Gender Classification from Facial Images.